<a href="https://colab.research.google.com/github/rocket-lab-dev/ARES-DBP/blob/main/code/HY_DBP_150M_BiMamba_DatasetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1: This cell is responsible for installing the necessary PyTorch version and Mamba libraries. It first attempts to install a specific PyTorch version compatible with CUDA, then uninstalls any potentially conflicting Mamba-related packages, and finally installs `causal-conv1d` and `mamba-ssm`.

In [1]:
# 1. Install correct PyTorch for Colab's CUDA 12.1
!pip install torch==2.4.0 --index-url https://lnkd.in/g_8dBghS

# 2. Uninstall any broken installs
!pip uninstall -y mamba-ssm causal-conv1d

# 3. Install Mamba using the PyTorch we just installed
!pip install causal-conv1d mamba-ssm --no-build-isolation

Looking in indexes: https://lnkd.in/g_8dBghS
ERROR: Could not find a version that satisfies the requirement torch==2.4.0 (from versions: none)
ERROR: No matching distribution found for torch==2.4.0
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 16.4 MB/s eta 0:00:00
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl size=254017554 sha256=93fc400b847e9b8ffc1dd3a90bc30d896893b90f056aef49b91a8a7eab78a229
  Stored in directory: /root/.cache/pip/wheels/98/4a/75/b24971cff4599825b16b612f08fbd2e60a2c336a56e081a3c8
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=533592144 sha256=d66a3c4c94a693e02d341cace7a6af0b72177b6afa655a25e3a6505130a68cbf
  Stored in directory: /root/.cache/pip/wheels/28/83/54/d45107838fec575b93f5d723f56351c

Cell 1: This code cell implements the 'CASE-B 150M' model, combining ESM-2 with a Bi-Mamba layer. It configures hardware resources, mounts Google Drive, defines helper functions for metrics and FASTA data loading, and specifies the `BiMambaBlock` and `CaseB_Model` architectures. The training involves a 5-fold stratified cross-validation, and the cell concludes with an independent test phase where predictions are ensembled using soft voting.

In [ ]:
# 导入 biopython 核心
!pip install biopython

In [ ]:
# -*- coding: utf-8 -*-
# ARES-DBP Case-B: ESM-2 150M + Bi-Mamba (Ablation without AttnRes)
# 🚀 Optimized for A100 80GB (Logging, Persistence, Hyper-tuned)

import os, gc, shutil, builtins, torch, random, datetime
import torch.nn as nn, pandas as pd, numpy as np
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, EsmModel, get_cosine_schedule_with_warmup, logging
from peft import LoraConfig, get_peft_model
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix, roc_auc_score, precision_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from Bio import SeqIO

try:
    from mamba_ssm import Mamba
except ImportError:
    print("❌ 核心算子 Mamba 未能正确安装，请执行: pip install causal-conv1d>=1.2.0 mamba-ssm")

# ==========================================
# 1. 硬件级优化与全链路持久化目录配置
# ==========================================
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

builtins.print("🔄 正在挂载 Google Drive...")
drive.mount('/content/drive')

WORKSPACE_ROOT = "/content/drive/MyDrive"
DATA_ROOT = f"{WORKSPACE_ROOT}/Dataset"
PROJECT_ROOT = f"{WORKSPACE_ROOT}/ARES_CaseB_150M_V2_Project"

# 严格的业务目录划分
MODEL_DIR = f"{PROJECT_ROOT}/models"
RESULT_DIR = f"{PROJECT_ROOT}/results"
LOG_DIR = f"{PROJECT_ROOT}/logs"
HF_CACHE = f"{WORKSPACE_ROOT}/cache_huggingface"
LOCAL_CKPT = "/content/checkpoints" # 挂在 Colab 本地，速度最快

for d in[MODEL_DIR, RESULT_DIR, LOG_DIR, LOCAL_CKPT, HF_CACHE]:
    os.makedirs(d, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
logging.set_verbosity_error()

# 🌟 双写日志引擎
LOG_FILE = f"{LOG_DIR}/CaseB_150M_run.log"
def dual_print(*args, **kwargs):
    builtins.print(*args, **kwargs)
    if kwargs.get("end") == "" or kwargs.get("end") == "\r": return
    msg = " ".join(map(str, args))
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {msg}\n")
print = dual_print

print("\n" + "="*80)
print(f"🚀 新任务启动: Case B (ESM-2 150M + Bi-Mamba) | 日志: {LOG_FILE}")
print("="*80)

# ==========================================
# 2. 超参数最优配置 (A100 强基线)
# ==========================================
MODEL_ID = "facebook/esm2_t30_150M_UR50D"
EMBED_DIM = 640
MAX_LEN = 1024

BATCH_SIZE = 128      # A100 大显存极速训练
EPOCHS = 10
WARMUP_RATIO = 0.1

# 非对称学习率：Mamba 头比 ESM 底座更需要强驱动力
LR_BACKBONE = 2e-4
LR_HEAD = 5e-4
WEIGHT_DECAY = 0.05   # 防止小样本过拟合

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
seed_everything(42)

# ==========================================
# 3. 评测引擎与内存驻留数据加载
# ==========================================
def calculate_all_metrics(y_true, y_prob):
    y_pred = (np.array(y_prob) > 0.5).astype(int)
    if len(np.unique(y_true)) < 2: return {"MCC": 0, "ACC": accuracy_score(y_true, y_pred)}
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "ACC": accuracy_score(y_true, y_pred),
        "PRE": precision_score(y_true, y_pred, zero_division=0),
        "SN": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "SP": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "MCC": matthews_corrcoef(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob)
    }

def load_fasta_to_ram(file_path, tokenizer):
    sequences, labels = [],[]
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    print(f"📂 正在解析数据源: {file_path}")
    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper()
        if not all(c in valid_aa for c in seq) or not (50 <= len(seq) <= MAX_LEN):
            continue
        try:
            label = int(record.id.split('_')[-1])
            sequences.append(seq)
            labels.append(label)
        except: continue

    print(f"🧠 正在全量分词并驻留内存...")
    enc = tokenizer(sequences, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors="pt")
    return enc['input_ids'], enc['attention_mask'], torch.tensor(labels, dtype=torch.float)

class StaticRAMDataset(Dataset):
    def __init__(self, ids, mask, labels):
        self.ids, self.mask, self.labels = ids, mask, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.labels[i]

# ==========================================
# 4. 架构定义: Case B (ESM-2 + Bi-Mamba)
# ==========================================
class BiMambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
    def forward(self, x):
        f = self.mamba_fwd(x)
        b = self.mamba_bwd(x.flip(1)).flip(1)
        return (f + b) / 2  # 平均池化特征

class ARES_CaseB_Model(nn.Module):
    def __init__(self):
        super().__init__()
        base = EsmModel.from_pretrained(MODEL_ID, cache_dir=HF_CACHE, torch_dtype=torch.bfloat16,
                                        device_map="cuda", attn_implementation="sdpa")
        base.gradient_checkpointing_disable()

        # 深度 LoRA
        lora_cfg = LoraConfig(r=32, lora_alpha=64, target_modules=["query", "key", "value", "dense"], lora_dropout=0.1)
        self.encoder = get_peft_model(base, lora_cfg)

        self.neck = BiMambaBlock(EMBED_DIM)

        # 缺少了 AttnRes 模块，直接接分类器 (Dropout 提升防过拟合)
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM, 512), nn.GELU(), nn.Dropout(0.4), nn.Linear(512, 1)
        ).to(torch.bfloat16)

    def forward(self, ids, mask):
        # 构建 4D 掩码适配 SDPA 加速
        ext_mask = mask.view(ids.shape[0], 1, 1, -1).to(torch.bfloat16)
        ext_mask = (1.0 - ext_mask) * -10000.0

        # 1. 进化语义提取
        esm_out = self.encoder(ids, attention_mask=ext_mask).last_hidden_state

        # 2. 时序语境提取 (特征在此处容易被稀释)
        mamba_out = self.neck(esm_out)

        # 3. 掩码池化与分类
        mask_f = mask.unsqueeze(-1).float()
        pooled = (mamba_out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled)

# ==========================================
# 5. 五折交叉验证 (全监控落盘)
# ==========================================
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
    f_ids, f_mask, f_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Train.fasta"), tokenizer)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_process_logs =[]

    for fold, (t_idx, v_idx) in enumerate(skf.split(f_ids, f_labels)):
        print(f"\n🚀 开始 Fold {fold+1}/5 | Case-B (ESM+Bi-Mamba)")
        train_ld = DataLoader(StaticRAMDataset(f_ids[t_idx], f_mask[t_idx], f_labels[t_idx]), batch_size=BATCH_SIZE, shuffle=True)
        val_ld = DataLoader(StaticRAMDataset(f_ids[v_idx], f_mask[v_idx], f_labels[v_idx]), batch_size=BATCH_SIZE)

        model = ARES_CaseB_Model().to("cuda")

        # 分组赋予学习率
        optimizer = torch.optim.AdamW([
            {'params': model.encoder.parameters(), 'lr': LR_BACKBONE},
            {'params': model.neck.parameters(), 'lr': LR_HEAD},
            {'params': model.classifier.parameters(), 'lr': LR_HEAD}
        ], weight_decay=WEIGHT_DECAY)

        total_steps = len(train_ld) * EPOCHS
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
        loss_fn = nn.BCEWithLogitsLoss()
        best_fold_mcc = -1

        for epoch in range(EPOCHS):
            model.train()
            for ids, mask, labels in tqdm(train_ld, desc=f"Fold {fold+1} Ep {epoch+1}"):
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                    loss = loss_fn(logits, labels.to("cuda"))

                loss.backward()
                # 【核心】：保护 Mamba 防止梯度爆炸
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

            # 验证评估
            model.eval()
            y_p, y_t =[],[]
            with torch.no_grad():
                for ids, mask, labels in val_ld:
                    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                        logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                    y_p.extend(torch.sigmoid(logits).cpu().float().numpy())
                    y_t.extend(labels.numpy())

            m = calculate_all_metrics(y_t, y_p)
            m.update({"Fold": fold+1, "Epoch": epoch+1})
            cv_process_logs.append(m)
            print(f"--> Val MCC: {m['MCC']:.4f} | AUC: {m['AUC']:.4f} | Sn: {m['SN']:.4f} | Sp: {m['SP']:.4f}")

            # 模型持久化：存放到挂载的 /models 文件夹中
            if m['MCC'] > best_fold_mcc:
                best_fold_mcc = m['MCC']
                local_p = f"{LOCAL_CKPT}/caseB_f{fold+1}.pt"
                target_p = os.path.join(MODEL_DIR, f"caseB_f{fold+1}.pt")
                torch.save(model.state_dict(), local_p)
                shutil.copy2(local_p, target_p)

        del model; torch.cuda.empty_cache(); gc.collect()

    # 存储 5折中间结果
    cv_csv_path = os.path.join(RESULT_DIR, "CaseB_CV_Intermediate_Logs.csv")
    pd.DataFrame(cv_process_logs).to_csv(cv_csv_path, index=False)
    print(f"✅ 5折验证完毕，过程日志已存入: {cv_csv_path}")

    # ==========================================
    # 6. Test.fasta 独立验证与预测结果落盘
    # ==========================================
    print("\n🔥 开始 Test.fasta 独立测试与集成推理...")
    t_ids, t_mask, t_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Test.fasta"), tokenizer)
    test_ld = DataLoader(StaticRAMDataset(t_ids, t_mask, t_labels), batch_size=BATCH_SIZE)

    all_fold_probs = []
    independent_test_metrics =[]
    y_true_test = t_labels.numpy()

    for fold in range(1, 6):
        print(f"正在加载 Fold {fold} 权重进行推理...")
        model = ARES_CaseB_Model().to("cuda")
        model.load_state_dict(torch.load(os.path.join(MODEL_DIR, f"caseB_f{fold}.pt")))
        model.eval()

        f_p =[]
        with torch.no_grad():
            for ids, mask, _ in test_ld:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                f_p.extend(torch.sigmoid(logits).cpu().float().numpy())

        all_fold_probs.append(f_p)
        m_fold = calculate_all_metrics(y_true_test, f_p)
        m_fold.update({"Eval_Type": f"Fold_{fold}"})
        independent_test_metrics.append(m_fold)
        del model; torch.cuda.empty_cache(); gc.collect()

    # 6.1 集成计算 (Soft Voting)
    ens_probs = np.mean(all_fold_probs, axis=0)
    ens_metrics = calculate_all_metrics(y_true_test, ens_probs)
    ens_metrics.update({"Eval_Type": "Ensemble_Final"})

    # 6.2 独立评测汇总表落盘
    final_report = pd.DataFrame(independent_test_metrics)
    final_report = pd.concat([final_report, pd.DataFrame([ens_metrics])], ignore_index=True)
    summary_csv = os.path.join(RESULT_DIR, "CaseB_Independent_Test_Summary.csv")
    final_report.to_csv(summary_csv, index=False)

    # 6.3 【新增】明细预测结果落盘 (方便后续计算混淆矩阵和画 ROC)
    df_preds = pd.DataFrame({
        "Sequence_Index": range(len(y_true_test)),
        "True_Label": y_true_test,
    })
    for i in range(5):
        df_preds[f"Prob_Fold_{i+1}"] = all_fold_probs[i]
    df_preds["Prob_Ensemble"] = ens_probs
    df_preds["Pred_Class"] = (ens_probs > 0.5).astype(int)

    preds_csv = os.path.join(RESULT_DIR, "CaseB_Test_Predictions_Detail.csv")
    df_preds.to_csv(preds_csv, index=False)

    print("\n" + "🏆"*25 + "\n✨ Case-B (ESM+Mamba) 独立测试终表 ✨\n" + "🏆"*25)
    print(final_report.to_markdown(index=False))
    print(f"\n📁 汇总表已保存: {summary_csv}")
    print(f"📁 每个样本的详细预测概率已保存: {preds_csv}")
    print(f"📜 完整运行日志持久化至: {LOG_FILE}")

In [ ]:
 # -*- coding: utf-8 -*-
# CASE-B 150M


import os
import gc
import shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, EsmModel, get_cosine_schedule_with_warmup, logging
from peft import LoraConfig, get_peft_model
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from Bio import SeqIO

try:
    from mamba_ssm import Mamba
except ImportError:
    print("请确保已安装 mamba-ssm")

# ==========================================
# 1. 硬件资源配置
# ==========================================
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Dataset"
SAVE_DIR = "/content/drive/MyDrive/CaseB_150M_BiMamba"
HF_CACHE = "/content/drive/MyDrive/hf_cache"
LOCAL_CKPT = "/content/checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
logging.set_verbosity_error()

# 参数配置
MODEL_ID = "facebook/esm2_t30_150M_UR50D"
EMBED_DIM = 640
BATCH_SIZE = 64  # A100 下 Mamba 效率极高，可稳定运行
MAX_LEN = 1024
LR = 5e-5
EPOCHS = 5
WARMUP_RATIO = 0.1

# ==========================================
# 2. 极速数据处理 (RAM 驻留)
# ==========================================
def calculate_all_metrics(y_true, y_prob):
    y_pred = (np.array(y_prob) > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ACC": accuracy_score(y_true, y_pred),
        "SN": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "SP": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "AUC": roc_auc_score(y_true, y_prob)
    }

def load_fasta_to_ram(file_path, tokenizer):
    sequences, labels = [], []
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    print(f"正在读取并解析: {file_path}")
    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper()
        if not all(c in valid_aa for c in seq) or not (50 <= len(seq) <= MAX_LEN):
            continue
        try:
            label = int(record.id.split('_')[-1])
            sequences.append(seq)
            labels.append(label)
        except: continue

    print(f"正在全量分词至 230GB RAM...")
    enc = tokenizer(sequences, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors="pt")
    return enc['input_ids'], enc['attention_mask'], torch.tensor(labels, dtype=torch.float)

class StaticRAMDataset(Dataset):
    def __init__(self, ids, mask, labels):
        self.ids, self.mask, self.labels = ids, mask, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return self.ids[i], self.mask[i], self.labels[i]

# ==========================================
# 3. 架构定义: ESM-2 + Bi-Mamba
# ==========================================
class BiMambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # 正向 Mamba
        self.mamba_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        # 反向 Mamba
        self.mamba_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)

    def forward(self, x):
        # x: [B, L, D]
        out_fwd = self.mamba_fwd(x)
        # 反向处理
        x_bwd = torch.flip(x, dims=[1])
        out_bwd = self.mamba_bwd(x_bwd)
        out_bwd = torch.flip(out_bwd, dims=[1])
        # 融合 (相加平衡特征强度)
        return out_fwd + out_bwd

class CaseB_Model(nn.Module):
    def __init__(self):
        super().__init__()
        # ESM-2 150M Backbone
        base = EsmModel.from_pretrained(
            MODEL_ID, cache_dir=HF_CACHE,
            torch_dtype=torch.bfloat16,
            device_map="cuda",
            attn_implementation="sdpa"
        )
        base.gradient_checkpointing_disable()

        # LoRA 微调
        lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["query", "key", "value"], lora_dropout=0.05)
        self.encoder = get_peft_model(base, lora_cfg)

        # Bi-Mamba Layer
        self.bi_mamba = BiMambaBlock(EMBED_DIM)

        # 分类头
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM, 512),
            nn.GELU(),
            nn.Linear(512, 1)
        )

    def forward(self, ids, mask):
        # 1. ESM 编码
        esm_out = self.encoder(ids, attention_mask=mask).last_hidden_state
        # 2. Bi-Mamba 处理
        mamba_out = self.bi_mamba(esm_out)
        # 3. Masked Mean Pooling
        mask_f = mask.unsqueeze(-1).float()
        pooled = (mamba_out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled)

# ==========================================
# 4. 训练与 5 折交叉验证
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
f_ids, f_mask, f_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Train.fasta"), tokenizer)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_process_logs = []

for fold, (t_idx, v_idx) in enumerate(skf.split(f_ids, f_labels)):
    print(f"\n🚀 Fold {fold+1}/5 | Case-B (ESM+Bi-Mamba)")
    train_ld = DataLoader(StaticRAMDataset(f_ids[t_idx], f_mask[t_idx], f_labels[t_idx]), batch_size=BATCH_SIZE, shuffle=True)
    val_ld = DataLoader(StaticRAMDataset(f_ids[v_idx], f_mask[v_idx], f_labels[v_idx]), batch_size=BATCH_SIZE)

    model = CaseB_Model().to("cuda")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    total_steps = len(train_ld) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    loss_fn = nn.BCEWithLogitsLoss()
    best_fold_mcc = -1

    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, labels in tqdm(train_ld, desc=f"Fold {fold+1} Ep {epoch+1}"):
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                loss = loss_fn(logits, labels.to("cuda"))
            loss.backward()
            optimizer.step()
            scheduler.step()

        # 验证
        model.eval()
        y_p, y_t = [], []
        with torch.no_grad():
            for ids, mask, labels in val_ld:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                y_p.extend(torch.sigmoid(logits).cpu().float().numpy())
                y_t.extend(labels.numpy())

        m = calculate_all_metrics(y_t, y_p)
        m.update({"Fold": fold+1, "Epoch": epoch+1})
        cv_process_logs.append(m)
        print(f"-> Val MCC: {m['MCC']:.4f}")

        if m['MCC'] > best_fold_mcc:
            best_fold_mcc = m['MCC']
            torch.save(model.state_dict(), f"{LOCAL_CKPT}/caseB_best_fold_{fold}.pt")
            shutil.copy2(f"{LOCAL_CKPT}/caseB_best_fold_{fold}.pt", os.path.join(SAVE_DIR, f"caseB_best_fold_{fold}.pt"))

    del model; torch.cuda.empty_cache(); gc.collect()

pd.DataFrame(cv_process_logs).to_csv(os.path.join(SAVE_DIR, "caseB_cv_process_logs.csv"), index=False)

# ==========================================
# 5. Test.fasta 独立验证与集成
# ==========================================
print("\n🔥 开始 Test.fasta 独立测试...")
t_ids, t_mask, t_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Test.fasta"), tokenizer)
test_ld = DataLoader(StaticRAMDataset(t_ids, t_mask, t_labels), batch_size=BATCH_SIZE)

all_fold_probs = []
independent_test_metrics = []
y_true_test = t_labels.numpy()

for fold in range(5):
    print(f"正在加载 Fold {fold} 推理...")
    model = CaseB_Model().to("cuda")
    model.load_state_dict(torch.load(f"{LOCAL_CKPT}/caseB_best_fold_{fold}.pt"))
    model.eval()

    f_p = []
    with torch.no_grad():
        for ids, mask, _ in tqdm(test_ld, desc=f"Fold {fold} 推理"):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
            f_p.extend(torch.sigmoid(logits).cpu().float().numpy())
    all_fold_probs.append(f_p)

    m_fold = calculate_all_metrics(y_true_test, f_p)
    m_fold.update({"Fold": fold})
    independent_test_metrics.append(m_fold)
    del model; torch.cuda.empty_cache(); gc.collect()

# 集成 (Soft Voting)
ens_probs = np.mean(all_fold_probs, axis=0)
ens_metrics = calculate_all_metrics(y_true_test, ens_probs)
ens_metrics.update({"Fold": "Ensemble_Final"})

# 保存最终报告
final_report = pd.DataFrame(independent_test_metrics)
final_report = pd.concat([final_report, pd.DataFrame([ens_metrics])], ignore_index=True)
final_report.to_csv(os.path.join(SAVE_DIR, "caseB_final_test_results.csv"), index=False)

print(f"\n✅ Case-B 完成！Ensemble MCC: {ens_metrics['MCC']:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.4 MB/s eta 0:00:00
Mounted at /content/drive
正在读取并解析: /content/drive/MyDrive/Dataset/Train.fasta
正在全量分词至 230GB RAM...

🚀 Fold 1/5 | Case-B (ESM+Bi-Mamba)


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Fold 1 Ep 1:   0%|          | 0/621 [00:00<?, ?it/s]

-> Val MCC: 0.5356


Fold 1 Ep 2:   0%|          | 0/621 [00:00<?, ?it/s]

...

Fold 3 推理:   0%|          | 0/196 [00:00<?, ?it/s]

正在加载 Fold 4 推理...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Fold 4 推理:   0%|          | 0/196 [00:00<?, ?it/s]


✅ Case-B 完成！Ensemble MCC: 0.5765


Cell 2: This code cell implements the 'CASE-C 150M' model. It handles hardware configuration, mounts Google Drive, defines utility functions for metric calculation and FASTA data loading, and implements the `AttnResBlock`, `BiMambaBlock`, and `CaseC_ARES_Model` architectures. The cell then proceeds with a 5-fold cross-validation training loop using a differential learning rate strategy and concludes with an ensemble inference on the `Test.fasta` dataset.

In [ ]:
# -*- coding: utf-8 -*-
# CASE-C 150M
# 导入 Mamba 与 ARES 核心组件
!pip install biopython

import os
import gc
import shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, EsmModel, get_cosine_schedule_with_warmup, logging
from peft import LoraConfig, get_peft_model
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from Bio import SeqIO

try:
    from mamba_ssm import Mamba
except ImportError:
    print("请确保已安装 mamba-ssm")

# ==========================================
# 1. 硬件资源配置与路径
# ==========================================
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Dataset"
SAVE_DIR = "/content/drive/MyDrive/Case_C_150M_ARES" # 修改保存目录
HF_CACHE = "/content/drive/MyDrive/hf_cache"
LOCAL_CKPT = "/content/checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
logging.set_verbosity_error()

# 参数配置
MODEL_ID = "facebook/esm2_t30_150M_UR50D"
EMBED_DIM = 640
BATCH_SIZE = 64
MAX_LEN = 1024
LR_BACKBONE = 5e-5 # LoRA 学习率
LR_NECK = 2e-4     # Mamba & AttnRes 学习率
EPOCHS = 6         # 增加 1 轮以稳定特征召回
WARMUP_RATIO = 0.1

# ==========================================
# 2. 辅助函数
# ==========================================
def calculate_all_metrics(y_true, y_prob):
    y_pred = (np.array(y_prob) > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ACC": accuracy_score(y_true, y_pred),
        "SN": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "SP": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "AUC": roc_auc_score(y_true, y_prob)
    }

def load_fasta_to_ram(file_path, tokenizer):
    sequences, labels = [], []
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    print(f"正在读取并解析: {file_path}")
    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper()
        if not all(c in valid_aa for c in seq) or not (50 <= len(seq) <= MAX_LEN):
            continue
        try:
            label = int(record.id.split('_')[-1])
            sequences.append(seq)
            labels.append(label)
        except: continue

    print(f"正在全量分词至 RAM (160GB+ 配置)...")
    enc = tokenizer(sequences, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors="pt")
    return enc['input_ids'], enc['attention_mask'], torch.tensor(labels, dtype=torch.float)

class StaticRAMDataset(Dataset):
    def __init__(self, ids, mask, labels):
        self.ids, self.mask, self.labels = ids, mask, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.labels[i]

# ==========================================
# 3. 架构定义: ESM-2 + Bi-Mamba + AttnRes (Case C)
# ==========================================
class AttnResBlock(nn.Module):
    """
    Case C 核心创新：注意力残差块
    Query: 来自 Mamba 的语境化特征
    Key/Value: 来自 ESM 的原始进化嵌入
    """
    def __init__(self, dim, heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, mamba_q, esm_kv):
        # 执行跨层特征回收
        res, _ = self.attn(query=mamba_q, key=esm_kv, value=esm_kv)
        return self.norm(mamba_q + res)

class BiMambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)

    def forward(self, x):
        out_fwd = self.mamba_fwd(x)
        x_bwd = torch.flip(x, dims=[1])
        out_bwd = self.mamba_bwd(x_bwd)
        out_bwd = torch.flip(out_bwd, dims=[1])
        return (out_fwd + out_bwd) / 2

class CaseC_ARES_Model(nn.Module):
    def __init__(self):
        super().__init__()
        # ESM-2 Backbone
        base = EsmModel.from_pretrained(
            MODEL_ID, cache_dir=HF_CACHE,
            torch_dtype=torch.bfloat16,
            device_map="cuda",
            attn_implementation="sdpa"
        )
        base.gradient_checkpointing_disable()

        # LoRA 微调
        lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["query", "key", "value"], lora_dropout=0.05)
        self.encoder = get_peft_model(base, lora_cfg)

        # Neck: Bi-Mamba
        self.bi_mamba = BiMambaBlock(EMBED_DIM)

        # Refiner: AttnRes (特征回收)
        self.refiner = AttnResBlock(EMBED_DIM)

        # 分类头
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, ids, mask):
        # 1. ESM 原始编码 (作为召回源)
        esm_raw = self.encoder(ids, attention_mask=mask).last_hidden_state

        # 2. Bi-Mamba 序列建模
        mamba_out = self.bi_mamba(esm_raw)

        # 3. AttnRes 特征召回 (Mamba Query -> ESM Raw Key/Value)
        fused_feat = self.refiner(mamba_out, esm_raw)

        # 4. Masked Mean Pooling
        mask_f = mask.unsqueeze(-1).float()
        pooled = (fused_feat * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled)

# ==========================================
# 4. 训练与 5 折交叉验证 (差异化学习率)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
f_ids, f_mask, f_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Train.fasta"), tokenizer)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_process_logs = []

for fold, (t_idx, v_idx) in enumerate(skf.split(f_ids, f_labels)):
    print(f"\n🚀 Fold {fold+1}/5 | Case-C (ARES-DBP: Mamba+AttnRes)")
    train_ld = DataLoader(StaticRAMDataset(f_ids[t_idx], f_mask[t_idx], f_labels[t_idx]), batch_size=BATCH_SIZE, shuffle=True)
    val_ld = DataLoader(StaticRAMDataset(f_ids[v_idx], f_mask[v_idx], f_labels[v_idx]), batch_size=BATCH_SIZE)

    model = CaseC_ARES_Model().to("cuda")

    # 设置差异化学习率
    optimizer = torch.optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': LR_BACKBONE},
        {'params': model.bi_mamba.parameters(), 'lr': LR_NECK},
        {'params': model.refiner.parameters(), 'lr': LR_NECK},
        {'params': model.classifier.parameters(), 'lr': LR_NECK}
    ], weight_decay=0.01)

    total_steps = len(train_ld) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    loss_fn = nn.BCEWithLogitsLoss()
    best_fold_mcc = -1

    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, labels in tqdm(train_ld, desc=f"Fold {fold+1} Ep {epoch+1}"):
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                loss = loss_fn(logits, labels.to("cuda"))
            loss.backward()
            optimizer.step()
            scheduler.step()

        # 验证
        model.eval()
        y_p, y_t = [], []
        with torch.no_grad():
            for ids, mask, labels in val_ld:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                y_p.extend(torch.sigmoid(logits).cpu().float().numpy())
                y_t.extend(labels.numpy())

        m = calculate_all_metrics(y_t, y_p)
        m.update({"Fold": fold+1, "Epoch": epoch+1})
        cv_process_logs.append(m)
        print(f"-> Val MCC: {m['MCC']:.4f} | SN: {m['SN']:.4f} | SP: {m['SP']:.4f}")

        if m['MCC'] > best_fold_mcc:
            best_fold_mcc = m['MCC']
            torch.save(model.state_dict(), f"{LOCAL_CKPT}/caseC_best_fold_{fold}.pt")
            shutil.copy2(f"{LOCAL_CKPT}/caseC_best_fold_{fold}.pt", os.path.join(SAVE_DIR, f"caseC_best_fold_{fold}.pt"))

    del model; torch.cuda.empty_cache(); gc.collect()

pd.DataFrame(cv_process_logs).to_csv(os.path.join(SAVE_DIR, "caseC_cv_process_logs.csv"), index=False)

# ==========================================
# 5. Test.fasta 集成推理 (产出 SOTA 报告)
# ==========================================
print("\n🔥 开始 Test.fasta 终极集成测试...")
t_ids, t_mask, t_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Test.fasta"), tokenizer)
test_ld = DataLoader(StaticRAMDataset(t_ids, t_mask, t_labels), batch_size=BATCH_SIZE)

all_fold_probs = []
independent_test_metrics = []
y_true_test = t_labels.numpy()

for fold in range(5):
    print(f"加载 Fold {fold} 权重进行推理...")
    model = CaseC_ARES_Model().to("cuda")
    model.load_state_dict(torch.load(f"{LOCAL_CKPT}/caseC_best_fold_{fold}.pt"))
    model.eval()

    f_p = []
    with torch.no_grad():
        for ids, mask, _ in tqdm(test_ld, desc=f"Fold {fold} 推理"):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
            f_p.extend(torch.sigmoid(logits).cpu().float().numpy())
    all_fold_probs.append(f_p)

    m_fold = calculate_all_metrics(y_true_test, f_p)
    m_fold.update({"Fold": fold})
    independent_test_metrics.append(m_fold)
    del model; torch.cuda.empty_cache(); gc.collect()

# 集成 (Soft Voting)
ens_probs = np.mean(all_fold_probs, axis=0)
ens_metrics = calculate_all_metrics(y_true_test, ens_probs)
ens_metrics.update({"Fold": "Ensemble_Final"})

# 保存报告
final_report = pd.DataFrame(independent_test_metrics)
final_report = pd.concat([final_report, pd.DataFrame([ens_metrics])], ignore_index=True)
final_report.to_csv(os.path.join(SAVE_DIR, "caseC_final_test_results.csv"), index=False)

print("\n" + "="*40)
print(f"🚀 ARES-DBP Case-C 完成！")
print(f"Ensemble MCC: {ens_metrics['MCC']:.4f}")
print(f"Ensemble Sn : {ens_metrics['SN']:.4f}")
print(f"Ensemble Sp : {ens_metrics['SP']:.4f}")
print("="*40)

Mounted at /content/drive
正在读取并解析: /content/drive/MyDrive/Dataset/Train.fasta
正在全量分词至 RAM (160GB+ 配置)...

🚀 Fold 1/5 | Case-C (ARES-DBP: Mamba+AttnRes)


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Fold 1 Ep 1:   0%|          | 0/621 [00:00<?, ?it/s]

-> Val MCC: 0.5649 | SN: 0.6532 | SP: 0.8928


Fold 1 Ep 2:   0%|          | 0/621 [00:00<?, ?it/s]

...

Fold 3 推理:   0%|          | 0/196 [00:00<?, ?it/s]

加载 Fold 4 权重进行推理...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Fold 4 推理:   0%|          | 0/196 [00:00<?, ?it/s]


🚀 ARES-DBP Case-C 完成！
Ensemble MCC: 0.5974
Ensemble Sn : 0.6793
Ensemble Sp : 0.9002


Cell 3: This code cell implements the 'CASE-C 650M' version of the model, which is a scaled-up version of the previous 'CASE-C 150M'. It adjusts parameters like `MODEL_ID` to `facebook/esm2_t33_650M_UR50D`, increases `EMBED_DIM` to 1280, and reduces `BATCH_SIZE` to 32 due to higher memory demands. The model architecture, training, and 5-fold cross-validation process are similar to the 150M version but optimized for the larger model, concluding with an ensemble test on `Test.fasta`.

In [ ]:
# -*- coding: utf-8 -*-
# CASE-C 650M Version
!pip install biopython

import os
import gc
import shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, EsmModel, get_cosine_schedule_with_warmup, logging
from peft import LoraConfig, get_peft_model
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from Bio import SeqIO

try:
    from mamba_ssm import Mamba
except ImportError:
    print("请确保已安装 mamba-ssm")

# ==========================================
# 1. 硬件资源配置与路径
# ==========================================
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Dataset"
SAVE_DIR = "/content/drive/MyDrive/Case_C_650M_ARES" # 修改为 650M 专用目录
HF_CACHE = "/content/drive/MyDrive/hf_cache"
LOCAL_CKPT = "/content/checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
logging.set_verbosity_error()

# 参数配置：适配 650M 模型
MODEL_ID = "facebook/esm2_t33_650M_UR50D" # 升级为 650M 骨干网
EMBED_DIM = 1280 # 650M 的隐藏层维度为 1280
BATCH_SIZE = 32  # 650M 显存占用更高，A100 下建议 32 以保证稳定性
MAX_LEN = 1024
LR_BACKBONE = 5e-5
LR_NECK = 2e-4
EPOCHS = 6
WARMUP_RATIO = 0.1

# ==========================================
# 2. 辅助函数
# ==========================================
def calculate_all_metrics(y_true, y_prob):
    y_pred = (np.array(y_prob) > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ACC": accuracy_score(y_true, y_pred),
        "SN": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "SP": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "AUC": roc_auc_score(y_true, y_prob)
    }

def load_fasta_to_ram(file_path, tokenizer):
    sequences, labels = [], []
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    print(f"正在读取并解析: {file_path}")
    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper()
        if not all(c in valid_aa for c in seq) or not (50 <= len(seq) <= MAX_LEN):
            continue
        try:
            label = int(record.id.split('_')[-1])
            sequences.append(seq)
            labels.append(label)
        except: continue

    print(f"正在全量分词至 RAM (针对 650M 高维特征优化)...")
    enc = tokenizer(sequences, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors="pt")
    return enc['input_ids'], enc['attention_mask'], torch.tensor(labels, dtype=torch.float)

class StaticRAMDataset(Dataset):
    def __init__(self, ids, mask, labels):
        self.ids, self.mask, self.labels = ids, mask, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.labels[i]

# ==========================================
# 3. 架构定义: ESM-2 + Bi-Mamba + AttnRes (Case C)
# ==========================================
class AttnResBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, mamba_q, esm_kv):
        res, _ = self.attn(query=mamba_q, key=esm_kv, value=esm_kv)
        return self.norm(mamba_q + res)

class BiMambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)

    def forward(self, x):
        out_fwd = self.mamba_fwd(x)
        x_bwd = torch.flip(x, dims=[1])
        out_bwd = self.mamba_bwd(x_bwd)
        out_bwd = torch.flip(out_bwd, dims=[1])
        return (out_fwd + out_bwd) / 2

class CaseC_ARES_650M_Model(nn.Module):
    def __init__(self):
        super().__init__()
        # ESM-2 650M Backbone
        base = EsmModel.from_pretrained(
            MODEL_ID, cache_dir=HF_CACHE,
            torch_dtype=torch.bfloat16, # A100 必须使用 BF16
            device_map="cuda",
            attn_implementation="sdpa"
        )
        base.gradient_checkpointing_disable()

        # LoRA 配置：适配 1280 维
        lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["query", "key", "value"], lora_dropout=0.05)
        self.encoder = get_peft_model(base, lora_cfg)

        # Neck & Refiner (自动适配 EMBED_DIM=1280)
        self.bi_mamba = BiMambaBlock(EMBED_DIM)
        self.refiner = AttnResBlock(EMBED_DIM)

        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, ids, mask):
        # 1. ESM 原始编码 (1280维)
        esm_raw = self.encoder(ids, attention_mask=mask).last_hidden_state
        # 2. Bi-Mamba 处理
        mamba_out = self.bi_mamba(esm_raw)
        # 3. AttnRes 特征召回
        fused_feat = self.refiner(mamba_out, esm_raw)
        # 4. Pooling
        mask_f = mask.unsqueeze(-1).float()
        pooled = (fused_feat * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled)

# ==========================================
# 4. 训练与 5 折交叉验证
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
f_ids, f_mask, f_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Train.fasta"), tokenizer)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_process_logs = []

for fold, (t_idx, v_idx) in enumerate(skf.split(f_ids, f_labels)):
    print(f"\n🚀 Fold {fold+1}/5 | Case-C (ARES-DBP: 650M Version)")
    train_ld = DataLoader(StaticRAMDataset(f_ids[t_idx], f_mask[t_idx], f_labels[t_idx]), batch_size=BATCH_SIZE, shuffle=True)
    val_ld = DataLoader(StaticRAMDataset(f_ids[v_idx], f_mask[v_idx], f_labels[v_idx]), batch_size=BATCH_SIZE)

    model = CaseC_ARES_650M_Model().to("cuda")

    optimizer = torch.optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': LR_BACKBONE},
        {'params': model.bi_mamba.parameters(), 'lr': LR_NECK},
        {'params': model.refiner.parameters(), 'lr': LR_NECK},
        {'params': model.classifier.parameters(), 'lr': LR_NECK}
    ], weight_decay=0.01)

    total_steps = len(train_ld) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    loss_fn = nn.BCEWithLogitsLoss()
    best_fold_mcc = -1

    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, labels in tqdm(train_ld, desc=f"Fold {fold+1} Ep {epoch+1}"):
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                loss = loss_fn(logits, labels.to("cuda"))
            loss.backward()
            optimizer.step()
            scheduler.step()

        # 验证
        model.eval()
        y_p, y_t = [], []
        with torch.no_grad():
            for ids, mask, labels in val_ld:
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
                y_p.extend(torch.sigmoid(logits).cpu().float().numpy())
                y_t.extend(labels.numpy())

        m = calculate_all_metrics(y_t, y_p)
        m.update({"Fold": fold+1, "Epoch": epoch+1})
        cv_process_logs.append(m)
        print(f"-> Val MCC: {m['MCC']:.4f} | SN: {m['SN']:.4f} | SP: {m['SP']:.4f}")

        if m['MCC'] > best_fold_mcc:
            best_fold_mcc = m['MCC']
            torch.save(model.state_dict(), f"{LOCAL_CKPT}/caseC_650M_best_fold_{fold}.pt")
            shutil.copy2(f"{LOCAL_CKPT}/caseC_650M_best_fold_{fold}.pt", os.path.join(SAVE_DIR, f"caseC_650M_best_fold_{fold}.pt"))

    del model; torch.cuda.empty_cache(); gc.collect()

pd.DataFrame(cv_process_logs).to_csv(os.path.join(SAVE_DIR, "caseC_650M_cv_process_logs.csv"), index=False)

# ==========================================
# 5. Test.fasta 集成推理
# ==========================================
print("\n🔥 开始 Test.fasta 终极集成测试 (650M Scale)...")
t_ids, t_mask, t_labels = load_fasta_to_ram(os.path.join(DATA_ROOT, "Test.fasta"), tokenizer)
test_ld = DataLoader(StaticRAMDataset(t_ids, t_mask, t_labels), batch_size=BATCH_SIZE)

all_fold_probs = []
independent_test_metrics = []
y_true_test = t_labels.numpy()

for fold in range(5):
    print(f"加载 650M Fold {fold} 权重...")
    model = CaseC_ARES_650M_Model().to("cuda")
    model.load_state_dict(torch.load(f"{LOCAL_CKPT}/caseC_650M_best_fold_{fold}.pt"))
    model.eval()

    f_p = []
    with torch.no_grad():
        for ids, mask, _ in tqdm(test_ld, desc=f"Fold {fold} 推理"):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(ids.to("cuda"), mask.to("cuda")).squeeze(-1)
            f_p.extend(torch.sigmoid(logits).cpu().float().numpy())
    all_fold_probs.append(f_p)

    m_fold = calculate_all_metrics(y_true_test, f_p)
    m_fold.update({"Fold": fold})
    independent_test_metrics.append(m_fold)
    del model; torch.cuda.empty_cache(); gc.collect()

ens_probs = np.mean(all_fold_probs, axis=0)
ens_metrics = calculate_all_metrics(y_true_test, ens_probs)
ens_metrics.update({"Fold": "Ensemble_Final"})

final_report = pd.DataFrame(independent_test_metrics)
final_report = pd.concat([final_report, pd.DataFrame([ens_metrics])], ignore_index=True)
final_report.to_csv(os.path.join(SAVE_DIR, "caseC_650M_final_test_results.csv"), index=False)

print("\n" + "="*40)
print(f"🚀 ARES-DBP Case-C (650M) 完成！")
print(f"Ensemble MCC: {ens_metrics['MCC']:.4f}")
print(f"Ensemble Sn : {ens_metrics['SN']:.4f}")
print(f"Ensemble Sp : {ens_metrics['SP']:.4f}")
print("="*40)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在读取并解析: /content/drive/MyDrive/Dataset/Train.fasta
正在全量分词至 RAM (针对 650M 高维特征优化)...

🚀 Fold 1/5 | Case-C (ARES-DBP: 650M Version)


model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

Fold 1 Ep 1:   0%|          | 0/1241 [00:00<?, ?it/s]

-> Val MCC: 0.5912 | SN: 0.6723 | SP: 0.8998


...

-> Val MCC: 0.5882 | SN: 0.7032 | SP: 0.8798

🚀 Fold 5/5 | Case-C (ARES-DBP: 650M Version)


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

Fold 5 Ep 1:   0%|          | 0/1241 [00:00<?, ?it/s]

-> Val MCC: 0.5915 | SN: 0.6502 | SP: 0.9121


Fold 5 Ep 2:   0%|          | 0/1241 [00:00<?, ?it/s]

Cell 5: This code cell performs automated hyperparameter optimization (HPO) for the 'CASE-C 150M' model using Optuna. It leverages Colab A100's capabilities, defines an ESM-2 150M + Bi-Mamba + AttnRes architecture with dynamic hyperparameters, and sets up a robust data engine with dynamic padding. The `objective` function guides Optuna to search for optimal parameters (e.g., learning rates, Mamba state/conv dimensions, attention heads, dropout, batch size, positive weight), reporting the best MCC score and hyperparameters after `N_TRIALS`.

!pip install causal-conv1d>=1.2.0 mamba-ssm

In [ ]:
!pip install causal-conv1d>=1.2.0 mamba-ssm

In [ ]:
# -*- coding: utf-8 -*-
# ==========================================
# 硬件环境：Colab A100 (80GB VRAM) + 160GB RAM
# 核心架构：ESM-2 150M + Bi-Mamba + AttnRes (Case-C)
# 极限加速：动态 Padding + 异步预取 + Flash Attention 2
# 任务目标：5折交叉验证与独立测试
# ==========================================
!pip install causal-conv1d>=1.2.0 mamba-ssm biopython peft transformers
import os
import gc
import shutil
import builtins
import threading
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, EsmModel, get_cosine_schedule_with_warmup, logging
from peft import LoraConfig, get_peft_model
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from Bio import SeqIO

try:
    from mamba_ssm import Mamba
except ImportError:
    raise ImportError("请先执行: !pip install causal-conv1d>=1.2.0 mamba-ssm")

# 🚀 A100 专属底层算子加速
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# ==========================================
# 1. 路径与全局配置
# ==========================================
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

WORKSPACE_ROOT = "/content/drive/MyDrive"
DATA_ROOT = f"{WORKSPACE_ROOT}/Dataset"
PROJECT_ROOT = f"{WORKSPACE_ROOT}/Case_C_150M_ARES_Fast"

HF_CACHE_DIR = f"{WORKSPACE_ROOT}/cache_huggingface"
os.environ['HF_HOME'] = HF_CACHE_DIR
logging.set_verbosity_error()

for d in ["models", "results", "logs"]:
    os.makedirs(f"{PROJECT_ROOT}/{d}", exist_ok=True)

LOG_FILE = f"{PROJECT_ROOT}/logs/CaseC_5Fold_Fast_run.log"
log_lock = threading.Lock()

def dual_print(*args, **kwargs):
    with log_lock:
        builtins.print(*args, **kwargs)
        if kwargs.get("end") in ["", "\r"]: return
        msg = " ".join(map(str, args))
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(f"[{timestamp}] {msg}\n")
print = dual_print

MODEL_ID = "facebook/esm2_t30_150M_UR50D"
EMBED_DIM = 640
MAX_LEN = 1024
BATCH_SIZE = 64  # 动态 Padding 下 64 毫无压力
EPOCHS = 15      # 提速后可以多跑几轮
LR_BACKBONE = 5e-5
LR_NECK = 2e-4

print("\n" + "="*80)
print(f"🚀 启动极速版 Case-C (150M+Mamba+ARES) 5折验证 | 日志: {LOG_FILE}")
print("="*80)

# ==========================================
# 2. 数据引擎 (动态 Padding 极速版)
# ==========================================
def load_fasta_to_ram_dynamic(file_path, tokenizer):
    sequences, labels = [],[]
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    print(f"📖 解析 FASTA: {file_path}")
    for record in SeqIO.parse(file_path, "fasta"):
        seq = str(record.seq).upper()
        if not all(c in valid_aa for c in seq) or not (50 <= len(seq) <= MAX_LEN):
            continue
        try:
            label = int(record.id.split('_')[-1])
            sequences.append(seq)
            labels.append(label)
        except: continue

    print(f"⚡ 正在进行无 Padding 分词 (驻留 160GB RAM)...")
    encodings = tokenizer(sequences, truncation=True, max_length=MAX_LEN, padding=False)

    input_ids =[torch.tensor(ids, dtype=torch.long) for ids in encodings['input_ids']]
    attention_mask =[torch.tensor(mask, dtype=torch.long) for ids in encodings['attention_mask']]
    labels_np = np.array(labels, dtype=np.float32)

    return input_ids, attention_mask, labels_np

class DynamicRAMDataset(Dataset):
    def __init__(self, ids_list, mask_list, labels):
        self.ids_list, self.mask_list, self.labels = ids_list, mask_list, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return self.ids_list[i], self.mask_list[i], self.labels[i]

def dynamic_collate_fn(batch):
    ids =[item[0] for item in batch]
    masks = [item[1] for item in batch]
    labels = torch.tensor([item[2] for item in batch], dtype=torch.float32)

    padded_ids = pad_sequence(ids, batch_first=True, padding_value=1)
    padded_masks = pad_sequence(masks, batch_first=True, padding_value=0)
    return padded_ids, padded_masks, labels

def calculate_all_metrics(y_true, y_prob):
    y_pred = (np.array(y_prob) > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ACC": accuracy_score(y_true, y_pred),
        "PRE": precision_score(y_true, y_pred, zero_division=0),
        "SN": tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        "SP": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

# ==========================================
# 3. 极速网络架构 (Case-C)
# ==========================================
class AttnResBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        # ⚠️ 核心提速：使用 batch_first=True，PyTorch 2.0+ 会自动在底层调用 Flash Attention
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, mamba_q, esm_kv, mask):
        # key_padding_mask 要求 True 为忽略区域 (Padding)
        key_padding_mask = (mask == 0)
        # need_weights=False 是触发 Flash Attention 的关键条件
        res, _ = self.attn(query=mamba_q, key=esm_kv, value=esm_kv,
                           key_padding_mask=key_padding_mask, need_weights=False)
        return self.norm(mamba_q + res)

class BiMambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)

    def forward(self, x):
        out_fwd = self.mamba_fwd(x)
        out_bwd = self.mamba_bwd(x.flip(dims=[1])).flip(dims=[1])
        return (out_fwd + out_bwd) / 2

class CaseC_ARES_FastModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = EsmModel.from_pretrained(
            MODEL_ID, cache_dir=HF_CACHE_DIR,
            torch_dtype=torch.bfloat16,
            device_map="cuda:0",
            attn_implementation="sdpa"
        )
        # 150M 模型在 A100 上绝对不需要梯度检查点，关闭以提速 30%
        base.gradient_checkpointing_disable()

        lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["query", "key", "value"], lora_dropout=0.05)
        self.encoder = get_peft_model(base, lora_cfg)

        self.bi_mamba = BiMambaBlock(EMBED_DIM)
        self.refiner = AttnResBlock(EMBED_DIM)

        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, ids, mask):
        esm_raw = self.encoder(ids, attention_mask=mask).last_hidden_state
        mamba_out = self.bi_mamba(esm_raw)
        fused_feat = self.refiner(mamba_out, esm_raw, mask)

        mask_f = mask.unsqueeze(-1).float()
        pooled = (fused_feat * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled).squeeze(-1)

# ==========================================
# 4. 训练与 5 折交叉验证
# ==========================================
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE_DIR)
    f_ids, f_mask, f_labels = load_fasta_to_ram_dynamic(os.path.join(DATA_ROOT, "Train.fasta"), tokenizer)

    # 动态计算正样本权重应对不平衡
    count_0, count_1 = np.bincount(f_labels.astype(int))
    pos_weight = torch.tensor([count_0 / count_1], dtype=torch.float32).to("cuda")
    print(f"⚖️ 类别权重已激活: pos_weight = {pos_weight.item():.4f}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_process_logs =[]

    for fold, (t_idx, v_idx) in enumerate(skf.split(np.zeros(len(f_labels)), f_labels)):
        print(f"\n🚀 Fold {fold+1}/5 | 极速动态 Padding 模式")

        t_ids_list = [f_ids[i] for i in t_idx]
        t_mask_list =[f_mask[i] for i in t_idx]
        t_labels_arr = f_labels[t_idx]

        v_ids_list = [f_ids[i] for i in v_idx]
        v_mask_list = [f_mask[i] for i in v_idx]
        v_labels_arr = f_labels[v_idx]

        # 🚀 核心提速：异步 DataLoader
        train_ld = DataLoader(
            DynamicRAMDataset(t_ids_list, t_mask_list, t_labels_arr),
            batch_size=BATCH_SIZE, shuffle=True, collate_fn=dynamic_collate_fn,
            num_workers=4, pin_memory=True, prefetch_factor=2
        )
        val_ld = DataLoader(
            DynamicRAMDataset(v_ids_list, v_mask_list, v_labels_arr),
            batch_size=BATCH_SIZE, shuffle=False, collate_fn=dynamic_collate_fn,
            num_workers=4, pin_memory=True
        )

        model = CaseC_ARES_FastModel().to("cuda")

        optimizer = torch.optim.AdamW([
            {'params': model.encoder.parameters(), 'lr': LR_BACKBONE},
            {'params': model.bi_mamba.parameters(), 'lr': LR_NECK},
            {'params': model.refiner.parameters(), 'lr': LR_NECK},
            {'params': model.classifier.parameters(), 'lr': LR_NECK}
        ], weight_decay=0.01, fused=True)

        total_steps = len(train_ld) * EPOCHS
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        best_fold_mcc = -1

        for epoch in range(EPOCHS):
            model.train()
            for ids, mask, labels in tqdm(train_ld, desc=f"Fold {fold+1} Ep {epoch+1}"):
                # 🚀 核心提速：异步拷贝至 GPU
                ids = ids.to("cuda", non_blocking=True)
                mask = mask.to("cuda", non_blocking=True)
                labels = labels.to("cuda", non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids, mask)
                    loss = loss_fn(logits, labels)
                loss.backward()
                optimizer.step()
                scheduler.step()

            # 验证
            model.eval()
            y_p, y_t = [],[]
            with torch.no_grad():
                for ids, mask, labels in val_ld:
                    ids = ids.to("cuda", non_blocking=True)
                    mask = mask.to("cuda", non_blocking=True)
                    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                        logits = model(ids, mask)
                    y_p.extend(torch.sigmoid(logits).cpu().float().numpy())
                    y_t.extend(labels.numpy())

            m = calculate_all_metrics(y_t, y_p)
            m.update({"Fold": fold+1, "Epoch": epoch+1})
            cv_process_logs.append(m)
            print(f"    -> Val MCC: {m['MCC']:.4f} | AUC: {m['AUC']:.4f}")

            if m['MCC'] > best_fold_mcc:
                best_fold_mcc = m['MCC']
                torch.save(model.state_dict(), f"{PROJECT_ROOT}/models/caseC_best_fold_{fold+1}.pt")

        del model, optimizer; torch.cuda.empty_cache(); gc.collect()

    pd.DataFrame(cv_process_logs).to_csv(os.path.join(PROJECT_ROOT, "results", "caseC_cv_process_logs.csv"), index=False)

    # ==========================================
    # 5. Test.fasta 独立测试与集成
    # ==========================================
    print("\n🔥 开始 Test.fasta 终极集成测试...")
    t_ids, t_mask, t_labels = load_fasta_to_ram_dynamic(os.path.join(DATA_ROOT, "Test.fasta"), tokenizer)
    test_ld = DataLoader(
        DynamicRAMDataset(t_ids, t_mask, t_labels),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=dynamic_collate_fn,
        num_workers=4, pin_memory=True
    )

    all_fold_probs = []
    independent_test_metrics =[]

    for fold in range(1, 6):
        print(f"🧪 加载 Fold {fold} 权重进行推理...")
        model = CaseC_ARES_FastModel().to("cuda")
        model.load_state_dict(torch.load(f"{PROJECT_ROOT}/models/caseC_best_fold_{fold}.pt"))
        model.eval()

        f_p =[]
        with torch.no_grad():
            for ids, mask, _ in tqdm(test_ld, desc=f"Fold {fold} 推理"):
                ids = ids.to("cuda", non_blocking=True)
                mask = mask.to("cuda", non_blocking=True)
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(ids, mask)
                f_p.extend(torch.sigmoid(logits).cpu().float().numpy())

        all_fold_probs.append(f_p)
        m_fold = calculate_all_metrics(t_labels, f_p)
        m_fold.update({"Phase": "Independent_Test", "Fold": fold})
        independent_test_metrics.append(m_fold)
        del model; torch.cuda.empty_cache(); gc.collect()

    # 集成 (Soft Voting)
    ens_probs = np.mean(all_fold_probs, axis=0)
    ens_metrics = calculate_all_metrics(t_labels, ens_probs)
    ens_metrics.update({"Phase": "Ensemble_Final", "Fold": "ALL"})

    final_report = pd.DataFrame(independent_test_metrics)
    final_report = pd.concat([final_report, pd.DataFrame([ens_metrics])], ignore_index=True)

    cols =['Phase', 'Fold', 'MCC', 'ACC', 'PRE', 'SN', 'SP', 'AUC', 'F1']
    final_report = final_report[cols]
    final_report.to_csv(os.path.join(PROJECT_ROOT, "results", "caseC_final_test_results.csv"), index=False)

    print("\n" + "="*40)
    print(f"🚀 ARES-DBP Case-C 极速版完成！")
    print(f"Ensemble MCC: {ens_metrics['MCC']:.4f}")
    print(f"Ensemble AUC: {ens_metrics['AUC']:.4f}")
    print("="*40)

  error: subprocess-exited-with-error
  
  × Building wheel for causal-conv1d (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for causal-conv1d
  error: subprocess-exited-with-error
  
  × Building wheel for mamba-ssm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for mamba-ssm
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (causal-conv1d, mamba-ssm)


ImportError: 请先执行: !pip install causal-conv1d>=1.2.0 mamba-ssm